# Natural and designed repeat modules

Compares HURDLER feasibility and IDT-orderable adaptive maximum fragment capacity for curated natural100, designed_all, and designed_primary100 collections. All downstream work uses the real middle copy of each inferred repeat region rather than its first copy. Every locally acceptable candidate is scored by the IDT API; rejection reasons reweight the GA before the same length is retried, and module count advances only after an explicit zero-violation result.

**Rules:** `legacy-optimized-v1`; **seed:** 42 unless explicitly noted.

In [ ]:
REPO = '/home/wendai/projects/hurdler/clone_repeat_protein'
RULE_PROFILE = 'legacy-optimized-v1'
CATALOG = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/periodic_v4/module_catalog_periodic_v4_middle.parquet'
RESULTS = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/periodic_v4/module_hurdler_results.parquet'
CONSTRUCTS = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/periodic_v4/optimized_constructs.parquet'
ADAPTIVE_TRACE = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/periodic_v4/adaptive_copy_search_trace.parquet'
FINAL_MODULE_SUMMARY = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/periodic_v4/module_final_summary.parquet'
HURDLER_FRACTIONS = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/periodic_v4/module_hurdler_usable_fraction.csv'
SCATTER_DATA = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/periodic_v4/module_length_copy_scatter_data.csv'
SUMMARY_DIR = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/periodic_v4'
FIGURE_DIR = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/figures/periodic_v4'

In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd

def sha256(path):
    path = Path(path)
    if not path.is_file(): return None
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

run_context = {'rule_profile': RULE_PROFILE, 'input_hashes': {}, 'row_counts': {}, 'filter_flow': [], 'limitations': []}

In [ ]:
catalog = pd.read_parquet(CATALOG)
results = pd.read_parquet(RESULTS)
constructs = pd.read_parquet(CONSTRUCTS)
assert catalog.selected_module_policy.eq('repeat-region-middle-unit-tie-earlier-v1').all()
assert catalog.unit_sequence.eq(catalog.selected_module_sequence).all()
run_context['input_hashes'] = {Path(path).name: sha256(path) for path in (CATALOG, RESULTS, CONSTRUCTS)}
results_compare = pd.concat([results, results.loc[results.in_designed_primary100].assign(collection='designed_primary100')], ignore_index=True)
constructs_compare = pd.concat([constructs, constructs.loc[constructs.in_designed_primary100].assign(collection='designed_primary100')], ignore_index=True)
ga_applicable = constructs_compare.loc[constructs_compare.ga_status.eq('passed')].copy()
ga_applicable['ga_score_improvement'] = ga_applicable.ga_initial_score - ga_applicable.ga_score
summary = (results_compare.groupby(['collection','plasmid']).success.agg(successes='sum', modules='count').reset_index())
summary['success_rate'] = summary.successes/summary.modules
capacity = constructs_compare.groupby(['collection','fragment_limit_bp']).agg(modules=('module_id','nunique'), median_mathematical_max=('mathematical_max_copies','median'), median_verified_max=('verified_max_copies','median'), local_ga_passes=('ga_local_constraints_passed','sum'), idt_api_scored=('idt_api_called','sum'), idt_rule_violations=('idt_violation_count','sum'), final_idt_passes=('final_passed','sum')).reset_index()
ga_summary = ga_applicable.groupby(['collection','fragment_limit_bp']).agg(constructs=('module_id','count'), ga_improved=('ga_improved','sum'), initial_repeated_re_sites=('ga_initial_repeated_re_site_excess','sum'), final_repeated_re_sites=('repeated_re_site_excess','sum'), repeated_re_sites_removed=('ga_repeated_re_site_excess_removed','sum'), median_score_improvement=('ga_score_improvement','median')).reset_index()
failure_modes = constructs_compare.groupby(['collection','fragment_limit_bp','optimization_status']).size().rename('modules').reset_index()
idt_summary = constructs_compare.groupby(['collection','idt_status']).agg(constructs=('module_id','size'), idt_rule_violations=('idt_violation_count','sum')).reset_index()
run_context['row_counts'] = {'catalog_modules': len(catalog), 'module_plasmid_rows': len(results), 'construct_cap_rows': len(constructs), 'ga_applicable': len(ga_applicable), 'natural100': int((catalog.collection=='natural100').sum()), 'designed_all': int((catalog.collection=='designed_all').sum()), 'designed_primary100': int(catalog.in_designed_primary100.sum())}
run_context['filter_flow'] = ['select the real middle repeat copy and preserve its exact variable residues', 'require middle-copy policy and selected-sequence equality', 'deduplicate exact units within collection', 'query eight plasmids', 'rank candidates by optimizability then frozen stable order', 'genetic refinement with repeated-RE-site fitness term', 'score every locally acceptable DNA with the IDT API before changing copy count', 'map IDT rejection reasons to GA weights and retry the same copy count', 'advance only after an explicit zero-violation/orderable result']
run_context['limitations'] = ['external vector/adapter deduction is configurable and is zero in this run', 'modules longer than 60AA remain in this report but not the 1-60AA curve', 'IDT scores require OAuth account credentials and are not imputed when unavailable', 'IDT orderability is a hard adaptive-search gate and can limit the achievable copy count even when all local constraints pass']
Path(SUMMARY_DIR).mkdir(parents=True, exist_ok=True)
summary.to_csv(Path(SUMMARY_DIR)/'module_success_summary.csv', index=False)
capacity.to_csv(Path(SUMMARY_DIR)/'module_capacity_summary.csv', index=False)
ga_summary.to_csv(Path(SUMMARY_DIR)/'module_ga_summary.csv', index=False)
failure_modes.to_csv(Path(SUMMARY_DIR)/'module_failure_modes.csv', index=False)
idt_summary.to_csv(Path(SUMMARY_DIR)/'module_idt_summary.csv', index=False)
summary

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
Path(FIGURE_DIR).mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
order = ['natural100','designed_all','designed_primary100']
fig, axes = plt.subplots(2, 2, figsize=(15, 11), facecolor='white')
axes = axes.ravel()
sns.barplot(data=summary, x='collection', y='success_rate', hue='plasmid', order=order, ax=axes[0], palette='colorblind')
sns.boxplot(data=constructs_compare, x='collection', y='verified_max_copies', hue='fragment_limit_bp', order=order, ax=axes[1], palette=['#4B2E83','#B7A57A'])
sns.boxplot(data=ga_applicable, x='collection', y='ga_repeated_re_site_excess_removed', hue='fragment_limit_bp', order=order, ax=axes[2], palette=['#2D7DD2','#F45D01'])
sns.countplot(data=constructs_compare, x='collection', hue='optimization_status', order=order, ax=axes[3], palette=['#4B2E83','#B7A57A','#85754D','#999999'])
for ax in axes: ax.tick_params(axis='x', rotation=20)
axes[0].set_title('HURDLER success by plasmid'); axes[1].set_title('Maximum verified full copies'); axes[2].set_title('Repeated RE sites removed by GA'); axes[3].set_title('Optimization outcomes (both caps)')
sns.despine(); fig.tight_layout()
for suffix in ('png','pdf'): fig.savefig(Path(FIGURE_DIR) / f'module_benchmark.{suffix}', dpi=300, facecolor='white')
fig

In [ ]:
capacity.merge(failure_modes.groupby(['collection','fragment_limit_bp']).modules.sum().rename('outcome_rows').reset_index(), on=['collection','fragment_limit_bp'])

In [ ]:
final_module_summary = pd.read_parquet(FINAL_MODULE_SUMMARY)
hurdler_fractions = pd.read_csv(HURDLER_FRACTIONS)
run_context['input_hashes'][Path(FINAL_MODULE_SUMMARY).name] = sha256(FINAL_MODULE_SUMMARY)
run_context['input_hashes'][Path(HURDLER_FRACTIONS).name] = sha256(HURDLER_FRACTIONS)
run_context['row_counts']['final_module_summary_rows'] = len(final_module_summary)
hurdler_fractions

In [ ]:
from IPython.display import Image, display
scatter_data = pd.read_csv(SCATTER_DATA)
scatter_figure = Path(FIGURE_DIR) / 'module_length_vs_max_orderable_copies.png'
assert set(scatter_data.module_type) == {'Natural', 'Designed'}
assert scatter_data.max_orderable_module_copies.ge(2).all()
assert scatter_figure.stat().st_size > 0
run_context['input_hashes'][Path(SCATTER_DATA).name] = sha256(SCATTER_DATA)
run_context['row_counts']['idt_orderable_repeat_scatter_modules'] = scatter_data.module_id.nunique()
display(Image(filename=str(scatter_figure)))
scatter_data.groupby('module_type').agg(modules=('module_id','nunique'), median_module_length_aa=('unit_length_aa','median'), median_orderable_copies=('max_orderable_module_copies','median')).reset_index()

In [ ]:
idt_violation_rows = []
for row in constructs_compare.itertuples(index=False):
    value = getattr(row, 'idt_violation_names_json', None)
    if not isinstance(value, str): continue
    for rule_name in json.loads(value):
        idt_violation_rows.append({'collection': row.collection, 'fragment_limit_bp': row.fragment_limit_bp, 'rule_name': rule_name})
idt_violations = pd.DataFrame(idt_violation_rows, columns=['collection','fragment_limit_bp','rule_name']).groupby(['collection','fragment_limit_bp','rule_name']).size().rename('violations').reset_index()
idt_violations.to_csv(Path(SUMMARY_DIR)/'module_idt_violations.csv', index=False)
idt_violations.sort_values(['collection','fragment_limit_bp','violations'], ascending=[True,True,False])

In [ ]:
idt_summary

In [ ]:
trace = pd.read_parquet(ADAPTIVE_TRACE)
run_context['input_hashes'][Path(ADAPTIVE_TRACE).name] = sha256(ADAPTIVE_TRACE)
trace_compare = pd.concat([trace, trace.loc[trace.in_designed_primary100].assign(collection='designed_primary100')], ignore_index=True)
constructs_compare['copy_change_vs_legacy'] = constructs_compare.verified_max_copies - constructs_compare.pre_adaptive_verified_max_copies
constructs_compare['reached_mathematical_bound'] = constructs_compare.verified_max_copies.eq(constructs_compare.adaptive_search_upper_bound_copies) & constructs_compare.final_passed.fillna(False)
constructs_compare['orderable_maximum'] = constructs_compare.final_passed.fillna(False) & constructs_compare.adaptive_orderable_passed.fillna(False) & constructs_compare.adaptive_boundary_proven.fillna(False)
adaptive_capacity = constructs_compare.groupby(['collection','fragment_limit_bp']).agg(modules=('module_id','nunique'), modules_with_maximum=('orderable_maximum','sum'), boundary_proven=('adaptive_boundary_proven','sum'), median_legacy_max=('pre_adaptive_verified_max_copies','median'), median_adaptive_max=('verified_max_copies','median'), maximum_adaptive_copies=('verified_max_copies','max'), recovered_copies=('copy_change_vs_legacy','sum'), reached_mathematical_bound=('reached_mathematical_bound','sum'), median_search_evaluations=('adaptive_search_evaluations','median'), median_winning_generations=('ga_generations','median')).reset_index()
adaptive_stops = constructs_compare.groupby(['collection','fragment_limit_bp','adaptive_stop_reason']).size().rename('constructs').reset_index()
trace_compare['idt_rejected'] = trace_compare.idt_explicit_pass.eq(False)
trace_compare['feedback_applied'] = trace_compare.idt_feedback_adjustments_json.fillna('[]').ne('[]')
generation_usage = trace_compare.groupby(['collection','fragment_limit_bp','phase','generations']).agg(evaluations=('module_id','size'), orderable_passes=('passed','sum'), idt_scored=('idt_api_called','sum'), idt_rejections=('idt_rejected','sum'), feedback_updates=('feedback_applied','sum')).reset_index()
rejection_rows = trace_compare.loc[trace_compare.idt_rejected, ['collection','fragment_limit_bp','copies','generations','idt_violation_names_json','idt_rule_scores_json','idt_feedback_adjustments_json']].copy()
rejection_rows['idt_reason'] = rejection_rows.idt_violation_names_json.map(json.loads)
idt_rejection_reasons = rejection_rows.explode('idt_reason').groupby(['collection','fragment_limit_bp','idt_reason']).size().rename('rejections').reset_index()
maximum_columns = ['module_id','collection','family','in_designed_primary100','fragment_limit_bp','unit_sequence','unit_length','mathematical_max_copies','pre_adaptive_verified_max_copies','verified_max_copies','adaptive_search_evaluations','adaptive_idt_scored_evaluations','ga_generations','adaptive_stop_reason','adaptive_boundary_proven','adaptive_boundary_evidence','adaptive_orderable_passed','plasmid','direction','site_i_position','site_ii_position','site_i_enzyme','site_ii_enzyme','site_iii_enzymes','dna_sequence','idt_status','idt_violation_count','idt_scored_sequence_sha256']
maximum_constructs = constructs.loc[constructs.final_passed.fillna(False) & constructs.adaptive_boundary_proven.fillna(False), maximum_columns].copy()
adaptive_capacity.to_csv(Path(SUMMARY_DIR)/'adaptive_maximum_summary.csv', index=False)
adaptive_stops.to_csv(Path(SUMMARY_DIR)/'adaptive_stop_reasons.csv', index=False)
generation_usage.to_csv(Path(SUMMARY_DIR)/'adaptive_generation_usage.csv', index=False)
idt_rejection_reasons.to_csv(Path(SUMMARY_DIR)/'adaptive_idt_rejection_reasons.csv', index=False)
rejection_rows.to_csv(Path(SUMMARY_DIR)/'adaptive_idt_feedback_trace.csv', index=False)
maximum_constructs.to_parquet(Path(SUMMARY_DIR)/'maximum_passed_constructs_notebook.parquet', index=False)
maximum_constructs.to_csv(Path(SUMMARY_DIR)/'maximum_passed_constructs_notebook.csv', index=False)
run_context['row_counts']['adaptive_trace_evaluations'] = len(trace)
run_context['row_counts']['idt_rejected_evaluations'] = int(trace_compare.idt_rejected.sum())
run_context['row_counts']['idt_feedback_updates'] = int(trace_compare.feedback_applied.sum())
run_context['row_counts']['maximum_constructs_with_dna'] = len(maximum_constructs)
run_context['row_counts']['boundary_proven_construct_caps'] = int(constructs.adaptive_boundary_proven.fillna(False).sum())
run_context['filter_flow'].extend(['set upper bound to floor((fragment cap - deduction)/(3*unit length))', 'binary search using the local plus IDT gate at 10 generations', 'after IDT rejection map its reasons to GA weights and retry the same copy count', 'from one copy above the short-search maximum, add exactly one module at a time', 'for each new copy count escalate generations through 10,20,40,60,80,100', 'accept a maximum only when it is IDT-orderable and reaches the mathematical cap, or the next copy remains non-orderable at 100 generations', 'return the explicitly orderable maximum count and exact DNA'])
run_context['limitations'].append('binary search treats short-generation orderability as monotone; the audited one-by-one phase and explicit 100-generation local plus IDT boundary proof determine the reported maximum')
adaptive_capacity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), facecolor='white')
long_capacity = constructs_compare.melt(id_vars=['collection','fragment_limit_bp'], value_vars=['pre_adaptive_verified_max_copies','verified_max_copies'], var_name='search_stage', value_name='copies')
sns.boxplot(data=long_capacity, x='collection', y='copies', hue='search_stage', order=order, ax=axes[0], palette=['#B7A57A','#4B2E83'])
sns.boxplot(data=constructs_compare.loc[constructs_compare.orderable_maximum], x='collection', y='ga_generations', hue='fragment_limit_bp', order=order, ax=axes[1], palette=['#4B2E83','#B7A57A'])
sns.barplot(data=adaptive_capacity, x='collection', y='reached_mathematical_bound', hue='fragment_limit_bp', order=order, ax=axes[2], palette=['#2D7DD2','#F45D01'])
for ax in axes: ax.tick_params(axis='x', rotation=20)
axes[0].set_title('Legacy maximum vs adaptive maximum'); axes[1].set_title('Generations used by winning DNA'); axes[2].set_title('Modules reaching fragment-length ceiling')
sns.despine(); fig.tight_layout()
for suffix in ('png','pdf'): fig.savefig(Path(FIGURE_DIR) / f'adaptive_maximum_search.{suffix}', dpi=300, facecolor='white')
fig

In [ ]:
run_context